In [ ]:
!pip -q install streamlit pyngrok google-generativeai pandas

In [ ]:
# Cell 1: Libraries
!pip -q install openai streamlit pyngrok pandas

import streamlit as st
import pandas as pd
from openai import OpenAI
import threading, time, os
from pyngrok import ngrok

# ngrok authtoken
ngrok.set_auth_token("3DNvmZUKBjOG2I2JblkFtkXPzcy_4np9kU1ayM1JY8DMXAJYh")

# app.py
app_code = '''
import streamlit as st
import pandas as pd
from openai import OpenAI

st.set_page_config(page_title="AI Insight Engine", layout="wide")
st.title("📊 AI Decision & Insight Engine")
st.markdown("Upload CSV – AI will generate insights, recommendations, and report.")

groq_key = st.text_input("Enter Groq API Key (from console.groq.com)", type="password")
uploaded_file = st.file_uploader("Upload CSV File", type=["csv"])

if uploaded_file and groq_key:
    df = pd.read_csv(uploaded_file)
    st.success("CSV Uploaded!")
    st.dataframe(df.head(10))

    num_cols = df.select_dtypes(include='number').columns.tolist()
    stats = {
        'rows': len(df),
        'cols': len(df.columns),
        'missing': df.isnull().sum().sum(),
        'numeric_summary': df[num_cols].describe().to_dict() if num_cols else {}
    }
    if len(num_cols) >= 2:
        stats['correlation'] = df[num_cols].corr().to_dict()
    else:
        stats['correlation'] = {}

    prompt = f"""
You are an AI data analyst. Dataset: {stats['rows']} rows, {stats['cols']} cols.
Missing values: {stats['missing']}
Numeric summary: {stats['numeric_summary']}
First 5 rows:
{df.head(5).to_string()}

Give:
1. Key Insights (3-5 bullet points)
2. Actionable Recommendations (2-4 items)
3. Full Analytical Report (headings: Overview, Findings, Recommendations, Conclusion)
"""

    if st.button("Generate AI Insights"):
        with st.spinner("Analyzing..."):
            try:
                client = OpenAI(api_key=groq_key, base_url="https://api.groq.com/openai/v1")
                response = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[{"role": "user", "content": prompt}]
                )
                answer = response.choices[0].message.content
                st.markdown(answer)
                st.download_button("📥 Download Report", answer, file_name="report.md")
            except Exception as e:
                st.error(f"Error: {e}")

st.markdown("---")
st.markdown("Built with Streamlit + Groq AI")
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py successfully created")

# Streamlit run
def run_streamlit():
    os.system("streamlit run app.py --server.port 8501")

threading.Thread(target=run_streamlit).start()
time.sleep(5)

# Ngrok tunnel
ngrok.kill()
public_url = ngrok.connect(8501)
print("🚀 Live URL:", public_url)
